# 06 — Balanced analytical perturbation design

Generates a deterministic balanced representative design and evaluates each complete pipeline against a prespecified reference analytical pipeline.


In [1]:
import os, sys, json, warnings, hashlib, platform, time, math, itertools, shutil
from pathlib import Path
os.environ.setdefault('OMP_NUM_THREADS','2'); os.environ.setdefault('OPENBLAS_NUM_THREADS','2'); os.environ.setdefault('MKL_NUM_THREADS','2')
warnings.filterwarnings('ignore')
def find_project_root(start=None):
    start=Path(start or os.getcwd()).resolve()
    for p in [start]+list(start.parents):
        if (p/'config.json').exists() and (p/'notebooks').exists() and (p/'data').exists(): return p
    for base in [Path('/content'),Path('/mnt/data')]:
        if base.exists():
            hits=list(base.rglob('PCOS_Phenotype_Robustness_Framework_v*_colab/config.json'))
            if hits: return hits[0].parent
    raise FileNotFoundError('Project root not found')
PROJECT_ROOT=find_project_root(); os.chdir(PROJECT_ROOT); print('PROJECT_ROOT=',PROJECT_ROOT)
# Colab/bootstrap dependencies. The notebooks are self-contained and do not import project .py files.
# In Google Colab this cell installs requirements automatically when imports are missing.
# Set PCOS_AUTO_INSTALL=0 to disable installation, or PCOS_AUTO_INSTALL=1 to force it.
def _maybe_install_requirements():
    import importlib.util, subprocess
    required = ['numpy','pandas','sklearn','matplotlib','scipy','openpyxl']
    missing = [m for m in required if importlib.util.find_spec(m) is None]
    force = os.environ.get('PCOS_AUTO_INSTALL','auto') == '1'
    disable = os.environ.get('PCOS_AUTO_INSTALL','auto') == '0'
    if (missing or force) and not disable:
        req = PROJECT_ROOT/'requirements.txt'
        if req.exists():
            print('Installing requirements because missing packages were detected:', missing)
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
_maybe_install_requirements()
CONFIG=json.load(open('config.json','r',encoding='utf-8'))
ACTIVE_MODE=os.environ.get('PCOS_PIPELINE_MODE',CONFIG.get('active_mode','PILOT_REVIEW_FAST'))
MODE=CONFIG['execution_modes'][ACTIVE_MODE]
print('ACTIVE_MODE=',ACTIVE_MODE, MODE)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer
from sklearn.decomposition import PCA, FastICA
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
ROOT=Path('.'); OUT=ROOT/'outputs'; DATA=ROOT/'data'; OUT.mkdir(exist_ok=True); DATA.mkdir(exist_ok=True)
MIN_CLUSTER_FRAC=float(CONFIG.get('min_cluster_fraction',0.05))
def ensure_dir(p): p=Path(p); p.mkdir(parents=True,exist_ok=True); return p
def save_df(df,path):
    path=Path(path); ensure_dir(path.parent)
    if path.suffix=='.parquet':
        try:
            df.to_parquet(path,index=False)
        except Exception as e:
            alt=path.with_suffix('.csv')
            df.to_csv(alt,index=False)
            print(f'[parquet fallback] Could not write {path.name} ({type(e).__name__}); wrote {alt.name} instead.')
    elif path.suffix=='.xlsx':
        df.to_excel(path,index=False)
    else:
        df.to_csv(path,index=False)
def read_df(path):
    path=Path(path)
    if path.suffix=='.parquet':
        if path.exists():
            try:
                return pd.read_parquet(path)
            except Exception as e:
                alt=path.with_suffix('.csv')
                if alt.exists():
                    print(f'[parquet fallback] Could not read {path.name} ({type(e).__name__}); reading {alt.name}.')
                    return pd.read_csv(alt)
                raise
        alt=path.with_suffix('.csv')
        if alt.exists():
            return pd.read_csv(alt)
        raise FileNotFoundError(f'Neither {path} nor fallback {alt} exists')
    if path.suffix in ['.xlsx','.xls']:
        return pd.read_excel(path)
    return pd.read_csv(path)
def numeric_clean(s):
    if pd.api.types.is_numeric_dtype(s): return pd.to_numeric(s, errors='coerce')
    x=s.astype(str).str.replace(',', '.', regex=False).str.replace(r'[^0-9eE+\-\.<>=]', '', regex=True).str.replace(r'^[<>]=?', '', regex=True).replace({'':np.nan,'nan':np.nan,'None':np.nan})
    return pd.to_numeric(x, errors='coerce')
def coalesce_numeric(df, candidates):
    cols=[c for c in candidates if c in df.columns]
    if not cols: return pd.Series(np.nan,index=df.index,dtype=float), None
    out=pd.Series(np.nan,index=df.index,dtype=float); src=[]
    for c in cols:
        v=numeric_clean(df[c]); mask=out.isna()&v.notna(); out[mask]=v[mask]
        if v.notna().sum()>0: src.append(f'{c} (n={int(v.notna().sum())})')
    return out, '; '.join(src[:4])
def cluster_valid(labels,n):
    vals,counts=np.unique(labels,return_counts=True)
    if len(vals)<2: return False,0.0,len(vals)
    mf=counts.min()/n
    return bool(mf>=MIN_CLUSTER_FRAC),float(mf),len(vals)
def fit_cluster(X,method='gmm',seed=0,k=2):
    n=len(X)
    if method=='kmeans': labels=KMeans(n_clusters=k,n_init=20,random_state=seed).fit_predict(X); unc=np.full(n,np.nan)
    elif method=='gmm':
        m=GaussianMixture(n_components=k,covariance_type='diag',reg_covar=1e-6,n_init=3,max_iter=100,random_state=seed).fit(X); labels=m.predict(X); unc=1-m.predict_proba(X).max(axis=1)
    elif method=='hierarchical': labels=AgglomerativeClustering(n_clusters=k,linkage='ward').fit_predict(X); unc=np.full(n,np.nan)
    elif method=='spectral': labels=SpectralClustering(n_clusters=k,assign_labels='kmeans',random_state=seed,affinity='nearest_neighbors',n_neighbors=min(15,max(2,n//20))).fit_predict(X); unc=np.full(n,np.nan)
    elif method=='consensus':
        # Fast consensus-by-medoids for Colab: generate an ensemble of k-means partitions
        # and choose the partition with the highest mean ARI to all other ensemble members.
        labs=[KMeans(n_clusters=k,n_init=5,random_state=seed+i).fit_predict(X) for i in range(8)]
        S=np.zeros((len(labs),len(labs)))
        for i in range(len(labs)):
            for j in range(i+1,len(labs)):
                S[i,j]=S[j,i]=adjusted_rand_score(labs[i],labs[j])
        labels=labs[int(np.argmax(S.mean(axis=1)))]
        unc=np.full(n, max(0.0, 1.0-float(S.mean())))
    else: raise ValueError(method)
    return np.asarray(labels).astype(int),unc
def transform_matrix(Xdf,imputation,scaling,reduction,seed=0):
    X=Xdf.copy()
    if imputation=='complete_case':
        keep=~X.isna().any(axis=1); X=X.loc[keep].copy(); ids=X.index.to_numpy(); arr=X.to_numpy(float)
    else:
        ids=X.index.to_numpy(); arr=X.to_numpy(float)
        if imputation=='median':
            arr=SimpleImputer(strategy='median').fit_transform(arr)
        elif imputation=='knn':
            arr=KNNImputer(n_neighbors=5).fit_transform(arr)
        elif imputation=='mice':
            base=SimpleImputer(strategy='median').fit_transform(arr)
            if (ACTIVE_MODE in ['MANUSCRIPT_FINAL','PUBLICATION_FULL','PUBLICATION_EXHAUSTIVE']) or os.environ.get('PCOS_TRUE_ITERATIVE_IMPUTERS','0')=='1':
                try:
                    arr=IterativeImputer(estimator=BayesianRidge(), max_iter=4, random_state=seed, initial_strategy='median', skip_complete=True).fit_transform(arr)
                except Exception:
                    arr=base
            else:
                # Colab-feasible deterministic MICE proxy used for the frozen manuscript run.
                # Optional true iterative imputation can be enabled with PCOS_TRUE_ITERATIVE_IMPUTERS=1.
                arr=base
        elif imputation=='missforest_like':
            base=SimpleImputer(strategy='median').fit_transform(arr)
            if (ACTIVE_MODE in ['MANUSCRIPT_FINAL','PUBLICATION_FULL','PUBLICATION_EXHAUSTIVE']) or os.environ.get('PCOS_TRUE_ITERATIVE_IMPUTERS','0')=='1':
                try:
                    est=ExtraTreesRegressor(n_estimators=10, random_state=seed, n_jobs=1, max_depth=6)
                    arr=IterativeImputer(estimator=est, max_iter=2, random_state=seed, initial_strategy='median', skip_complete=True).fit_transform(arr)
                except Exception:
                    arr=base
            else:
                # Colab-feasible MissForest-like proxy; KNN preserves local multivariate structure without very long runtime.
                try:
                    arr=KNNImputer(n_neighbors=7).fit_transform(arr)
                except Exception:
                    arr=base
        else: raise ValueError(imputation)
    if scaling=='zscore': arr=StandardScaler().fit_transform(arr)
    elif scaling=='robust': arr=RobustScaler().fit_transform(arr)
    elif scaling=='quantile': arr=QuantileTransformer(output_distribution='normal',n_quantiles=min(200,len(arr)),random_state=seed).fit_transform(arr)
    else: raise ValueError(scaling)
    if reduction=='none': Z=arr
    elif reduction.startswith('pca'): Z=PCA(n_components=float(reduction.replace('pca',''))/100,svd_solver='full',random_state=seed).fit_transform(arr)
    elif reduction=='ica': Z=FastICA(n_components=min(10,arr.shape[1],len(arr)-1),random_state=seed,max_iter=500,whiten='unit-variance').fit_transform(arr)
    else: raise ValueError(reduction)
    return Z,ids
def pairwise_coassignment(labels,ids,all_ids):
    M=np.full((len(all_ids),len(all_ids)),np.nan); loc={v:i for i,v in enumerate(all_ids)}; idx=[loc[i] for i in ids]
    C=(labels[:,None]==labels[None,:]).astype(float); M[np.ix_(idx,idx)]=C; return M


def variation_of_information(a,b):
    a=np.asarray(a); b=np.asarray(b); n=len(a)
    if n==0: return np.nan
    _,ai=np.unique(a,return_inverse=True); _,bi=np.unique(b,return_inverse=True)
    pa=np.bincount(ai)/n; pb=np.bincount(bi)/n
    H_a=-(pa*np.log2(pa+1e-12)).sum(); H_b=-(pb*np.log2(pb+1e-12)).sum()
    I=0.0
    for i in range(len(pa)):
        for j in range(len(pb)):
            pij=np.mean((ai==i)&(bi==j))
            if pij>0: I += pij*np.log2(pij/(pa[i]*pb[j])+1e-12)
    return float(H_a+H_b-2*I)

def coassignment_jaccard(a,b):
    a=np.asarray(a); b=np.asarray(b); n=len(a)
    if n<3: return np.nan
    A=(a[:,None]==a[None,:]); B=(b[:,None]==b[None,:])
    tri=np.triu_indices(n,1)
    A=A[tri]; B=B[tri]
    union=np.logical_or(A,B).sum()
    if union==0: return np.nan
    return float(np.logical_and(A,B).sum()/union)

def bootstrap_ci(x, n_boot=200, seed=42, q=(2.5,97.5)):
    x=np.asarray(pd.Series(x).dropna(),float)
    if len(x)<2: return (np.nan,np.nan)
    rng=np.random.default_rng(seed); vals=[]
    for _ in range(int(n_boot)):
        vals.append(np.median(rng.choice(x, size=len(x), replace=True)))
    return tuple(np.percentile(vals,q))


PROJECT_ROOT= /mnt/data/user_dataset_run/PCOS_Phenotype_Robustness_Framework_v4_final_colab
ACTIVE_MODE= PILOT_REVIEW_FAST {'seeds': [0, 1, 2], 'max_specs': 36, 'bootstrap_n': 75}


In [ ]:
out=ensure_dir(OUT/'06_pipeline_perturbation')
X=read_df(OUT/'03_preprocessing/phenotyping_matrix.csv').set_index('analysis_id')
full_mode=ACTIVE_MODE in ['MANUSCRIPT_FINAL','PUBLICATION_FULL','PUBLICATION_EXHAUSTIVE']
imps=CONFIG['grids']['full_imputation'] if full_mode else CONFIG['grids']['fast_imputation']
scales=CONFIG['grids']['full_scaling'] if full_mode else CONFIG['grids']['fast_scaling']
reds=CONFIG['grids']['full_reduction'] if full_mode else CONFIG['grids']['fast_reduction']
clusters=CONFIG['grids']['full_clustering'] if full_mode else CONFIG['grids']['fast_clustering']
seeds=MODE['seeds']
# Full candidate space, then deterministic balanced greedy thinning.
all_specs=[dict(imputation=i,scaling=s,reduction=r,clustering=c,seed=int(seed),k=2)
           for i in imps for s in scales for r in reds for c in clusters for seed in seeds]
max_specs=MODE.get('max_specs')
def balanced_thin(specs,n,seed=42):
    if n is None or n>=len(specs): return specs
    rng=np.random.default_rng(seed); remaining=list(range(len(specs))); selected=[]
    factors=['imputation','scaling','reduction','clustering','seed']
    counts={f:{v:0 for v in sorted({x[f] for x in specs},key=str)} for f in factors}
    # choose candidates maximizing underrepresented factor levels plus diversity
    while len(selected)<n and remaining:
        sample=remaining if len(remaining)<=2000 else list(rng.choice(remaining,2000,replace=False))
        best=None; bestscore=-1e9
        for idx in sample:
            sp=specs[idx]
            score=sum(1/(1+counts[f][sp[f]]) for f in factors)
            if selected:
                score += 0.05*min(sum(sp[f]!=specs[j][f] for f in factors) for j in selected)
            score += rng.uniform(0,1e-8)
            if score>bestscore: bestscore=score; best=idx
        selected.append(best); remaining.remove(best)
        for f in factors: counts[f][specs[best][f]]+=1
    return [specs[i] for i in selected]
specs=balanced_thin(all_specs,max_specs,CONFIG['random_seed'])
save_df(pd.DataFrame(specs),out/'predefined_balanced_pipeline_design.csv')
# reference analytical pipeline: comparator only, not asserted to be optimal
reference_spec=dict(imputation='median',scaling='zscore',reduction='pca80',clustering='gmm',seed=CONFIG['random_seed'],k=2)
Zref,ids_ref=transform_matrix(X,**{k:reference_spec[k] for k in ['imputation','scaling','reduction']},seed=reference_spec['seed'])
lab_ref,unc_ref=fit_cluster(Zref,reference_spec['clustering'],reference_spec['seed'],reference_spec['k'])
ref_map=dict(zip(ids_ref,lab_ref))
rows=[]; frames=[]; post=[]
for run_id,sp in enumerate(specs):
    try:
        Z,ids=transform_matrix(X,sp['imputation'],sp['scaling'],sp['reduction'],sp['seed'])
        labels,unc=fit_cluster(Z,sp['clustering'],sp['seed'],sp['k'])
        valid,mf,nc=cluster_valid(labels,len(labels))
        common=sorted(set(ids)&set(ids_ref)); m=dict(zip(ids,labels))
        a=[ref_map[i] for i in common]; b=[m[i] for i in common]
        ari=adjusted_rand_score(a,b) if len(common)>3 else np.nan
        nmi=normalized_mutual_info_score(a,b) if len(common)>3 else np.nan
        vi=variation_of_information(a,b) if len(common)>3 else np.nan
        jac=coassignment_jaccard(a,b) if len(common)>3 else np.nan
        sil=silhouette_score(Z,labels) if valid and nc>1 and len(Z)>nc else np.nan
        row={'run_id':run_id,**sp,'n_patients':len(ids),'n_common_reference':len(common),'n_clusters':nc,
             'min_cluster_fraction':mf,'valid_solution':valid,'ari_vs_reference':ari,'nmi_vs_reference':nmi,
             'variation_of_information_vs_reference':vi,'coassignment_jaccard_vs_reference':jac,
             'silhouette':sil,'mean_posterior_uncertainty':float(np.nanmean(unc)) if unc is not None else np.nan}
        rows.append(row)
        if valid:
            frames.append(pd.DataFrame({'analysis_id':ids,'run_id':run_id,'label':labels,**{k:sp[k] for k in sp}}))
            if unc is not None: post.append(pd.DataFrame({'analysis_id':ids,'run_id':run_id,'posterior_uncertainty':unc}))
    except Exception as e:
        rows.append({'run_id':run_id,**sp,'error':str(e),'valid_solution':False})
res=pd.DataFrame(rows); save_df(res,out/'pipeline_perturbation_results_all.csv')
valid=res[res.valid_solution.fillna(False)].copy(); save_df(valid,out/'pipeline_perturbation_results_valid_only.csv')
labtab=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame(); save_df(labtab,out/'valid_pipeline_patient_labels.csv')
if post: save_df(pd.concat(post,ignore_index=True),out/'posterior_uncertainty_by_patient_run.csv')
# Bootstrap CIs for global run-level metrics.
summary=[]
for metric in ['ari_vs_reference','nmi_vs_reference','variation_of_information_vs_reference','coassignment_jaccard_vs_reference','silhouette','mean_posterior_uncertainty']:
    if metric in valid:
        lo,hi=bootstrap_ci(valid[metric],MODE.get('bootstrap_n',200),CONFIG['random_seed'])
        summary.append({'metric':metric,'n_valid':len(valid),'median':valid[metric].median(),'q25':valid[metric].quantile(.25),'q75':valid[metric].quantile(.75),'bootstrap_median_ci_low':lo,'bootstrap_median_ci_high':hi})
metric_ci=pd.DataFrame(summary)
save_df(metric_ci,out/'manuscript_metric_summary.csv')
save_df(metric_ci,out/'bootstrap_confidence_intervals_run_metrics.csv')
for group in ['imputation','scaling','reduction','clustering','seed']:
    g=valid.groupby(group).agg(n=('ari_vs_reference','size'),median_ari=('ari_vs_reference','median'),median_nmi=('nmi_vs_reference','median'),median_jaccard=('coassignment_jaccard_vs_reference','median'),median_silhouette=('silhouette','median'),median_uncertainty=('mean_posterior_uncertainty','median')).reset_index()
    save_df(g,out/f'manuscript_summary_by_{group}.csv')
print('candidate',len(res),'valid',len(valid),'design levels:',{x:pd.DataFrame(specs)[x].value_counts().to_dict() for x in ['imputation','scaling','reduction','clustering','seed']})
